# Tu tabla Delta no es una tabla · demo W4TT

**Antes de empezar:** lakehouse por defecto adjunto, **Runtime 2.0**, y ejecutados los pasos 0 y 1 (generar datos es lo lento).
Para la parte 2, crear antes el modelo semántico Direct Lake `sales_dl` (desde el lakehouse: *New semantic model*) con las tablas `sales_clustered` y `sales_partitioned`.
En directo (4 minutos cada parte): parte 1, pasos 2 a 7 · parte 2, celdas D1 a D4.

In [ ]:
%pip install -q semantic-link-labs
%pip install lxml

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 8, Finished, Available, Finished, True)


[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: /nfs4/pyenv-ef59afe5-f060-448a-b4d7-6f0967304f1c/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 45.1 MB/s  0:00:00

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: /nfs4/pyenv-ef59afe5-f060-448a-b4d7-6f0967304f1c/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



## 0 · Parámetros y utilidades

In [ ]:
ROWS_PER_WRITE = 2_000_000   # 3 writes -> 6M rows
CUSTOMERS      = 5_000       # high cardinality on purpose
PROBE_CUSTOMER = 42

from pyspark.sql import functions as F
import json, time

def count_files(table):
    d = spark.sql(f"DESCRIBE DETAIL {table}").first()
    return d["numFiles"], round(d["sizeInBytes"] / 1024**2, 1)

def files_read(table, customer=PROBE_CUSTOMER):
    """Files Spark will actually open for a point query, after partition pruning and min/max skipping."""
    df = spark.table(table).where(F.col("customer_id") == customer)
    t = time.time(); n = df.count(); secs = round(time.time() - t, 2)
    return len(df.inputFiles()), n, secs

def gen(n):
    return (spark.range(n)
        .withColumn("order_id", F.expr("uuid()"))
        .withColumn("order_date", F.expr("date_sub(current_date(), cast(rand() * 730 as int))"))
        .withColumn("customer_id", (F.rand() * CUSTOMERS).cast("int"))
        .withColumn("amount", F.round(F.rand() * 500, 2))
        .drop("id"))

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 10, Finished, Available, Finished, False)

## 1 · Tres escrituras en una «tabla» (antes de empezar)

In [ ]:
spark.sql("DROP TABLE IF EXISTS sales_demo")
for i in range(3):
    gen(ROWS_PER_WRITE).write.format("delta").mode("overwrite" if i == 0 else "append").saveAsTable("sales_demo")
print("files, MB:", count_files("sales_demo"))

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 11, Finished, Available, Finished, False)

files, MB: (24, 239.4)


# Parte 1 · Lo que hay dentro de la tabla

## 2 · Es una carpeta de ficheros Parquet
Mirad los nombres: ni filas ni «tabla», solo ficheros.

In [ ]:
base = "Tables/sales_demo"
parquet = [f for f in notebookutils.fs.ls(base) if f.name.endswith(".parquet")]
print(f"{len(parquet)} parquet files")
for f in parquet[:5]:
    print(f"  {f.name}  {round(f.size/1024**2,1)} MB")

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 12, Finished, Available, Finished, False)

24 parquet files
  part-00000-6847f0d3-ef3a-41be-922e-4454db19b233-c000.snappy.parquet  10.0 MB
  part-00000-c24c0b78-db22-4e03-8285-34c653cc3e60-c000.snappy.parquet  10.0 MB
  part-00000-e12b790f-7d33-4874-9355-57830336003d-c000.snappy.parquet  10.0 MB
  part-00001-4ccd43b2-d559-4cf1-8848-0afabbf57d48-c000.snappy.parquet  10.0 MB
  part-00001-6881ba94-d56f-4cd5-9341-11b50b9df588-c000.snappy.parquet  10.0 MB


## 3 · …y un log que lleva la cuenta
Cada escritura es un commit en JSON. Mirad las acciones `add` y sus **estadísticas min/max**: con ellas se saltan ficheros.

In [ ]:
log = sorted(f.path for f in notebookutils.fs.ls(f"{base}/_delta_log") if f.name.endswith(".json"))
print("\n".join(p.split("/")[-1] for p in log))

first_add = next(json.loads(l)["add"] for l in notebookutils.fs.head(log[-1], 1_000_000).splitlines() if '"add"' in l)
print("\npath :", first_add["path"])
print("stats:", json.dumps(json.loads(first_add["stats"]), indent=2))

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 13, Finished, Available, Finished, False)

00000000000000000000.json
00000000000000000001.json
00000000000000000002.json

path : part-00000-c24c0b78-db22-4e03-8285-34c653cc3e60-c000.snappy.parquet
stats: {
  "numRecords": 250000,
  "minValues": {
    "amount": -0.0,
    "customer_id": 0,
    "order_date": "2024-11-02",
    "order_id": "00003f54-9f85-453b-8940-dabab112"
  },
  "maxValues": {
    "amount": 500.0,
    "customer_id": 4999,
    "order_date": "2026-11-01",
    "order_id": "ffffeea6-b9ee-4fde-b627-09562e06\u007f"
  },
  "nullCount": {
    "amount": 0,
    "customer_id": 0,
    "order_date": 0,
    "order_id": 0
  },
  "tightBounds": true
}


## 4 · El log es la tabla: historial y viaje en el tiempo

In [ ]:
display(spark.sql("DESCRIBE HISTORY sales_demo").select("version", "timestamp", "operation", "operationMetrics.numOutputRows"))
print("rows now      :", spark.table("sales_demo").count())
print("rows version 0:", spark.sql("SELECT COUNT(order_id) FROM sales_demo VERSION AS OF 0").first()[0])

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 51e04bd8-d01e-4a3a-ba30-b62e2c296e1a)

rows now      : 6000000
rows version 0: 2000000


1. ## 5 · La mala idea: `PARTITIONED BY customer_id`
Una carpeta por cliente: miles de ficheros diminutos. *Estamos con una F2, esto lo dejamos ejecutado y lo enseñamos, que así nos ahorramos editar el video ;).*

In [ ]:
spark.sql("DROP TABLE IF EXISTS sales_partitioned")
spark.sql("CREATE TABLE sales_partitioned USING DELTA PARTITIONED BY (customer_id) AS SELECT order_id, order_date, customer_id, amount FROM sales_demo DISTRIBUTE BY customer_id")
folders = len(notebookutils.fs.ls("Tables/sales_partitioned")) - 1   # minus _delta_log
files, mb = count_files("sales_partitioned")
print(f"{folders} folders, {files} files, {mb} MB  ->  avg {round(mb*1024/files)} KB per file")

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 15, Finished, Available, Finished, False)

5001 folders, 5000 files, 277.6 MB  ->  avg 57 KB per file


## 6 · Liquid Clustering: se declara una vez y OPTIMIZE sabe qué hacer
*Una F2 da para lo que da...*

In [ ]:
spark.sql("DROP TABLE IF EXISTS sales_clustered")
spark.sql("CREATE TABLE sales_clustered USING DELTA CLUSTER BY (customer_id) AS SELECT order_id, order_date, customer_id, amount FROM sales_demo")
spark.sql("OPTIMIZE sales_clustered")
display(spark.sql("DESCRIBE DETAIL sales_clustered").select("numFiles", "sizeInBytes", "clusteringColumns"))

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e0793036-3687-444a-b9cc-7d1252f1d137)

## 7 · La misma consulta, tres formas de colocar los ficheros
¿Cuántos ficheros abre `WHERE customer_id = 42`?

In [ ]:
rows = []
for t in ["sales_demo", "sales_partitioned", "sales_clustered"]:
    total, _ = count_files(t)
    read, n, secs = files_read(t)
    rows.append((t, total, read, n, secs))
display(spark.createDataFrame(rows, "table string, total_files int, files_read int, rows_found long, seconds double"))

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dd3fd147-3202-44c3-be22-982ed091985c)

## Extra · ¿Ha funcionado el clustering? `clusteringQuality()` (Scala, Runtime 2.0)
`avg_depth` cerca de **1**: una consulta puntual toca ~1 fichero.

In [ ]:
%%spark
import io.delta.tables.DeltaTable
display(DeltaTable.forName(spark, "sales_clustered").clusteringQuality())

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9ec87aa7-8fc3-42a7-bb56-ca06a7aa885e)


import io.delta.tables.DeltaTable


## Extra · Cambiar de idea es barato

In [ ]:
spark.sql("ALTER TABLE sales_clustered CLUSTER BY (order_date)")
display(spark.sql("DESCRIBE DETAIL sales_clustered").select("clusteringColumns", "numFiles"))
# spark.sql("OPTIMIZE sales_clustered FULL")   # full recluster: not in your daily pipeline

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0a2ea98a-d03c-48ca-a2fb-e4cc4a2e98fe)

# Parte 2 · Lo que ve Power BI

## D0 · Preparación (antes de empezar)
Modelo Direct Lake `sales_dl` sobre `sales_clustered` y `sales_partitioned`, en este mismo workspace.

In [ ]:
import sempy.fabric as fabric
import sempy_labs.directlake as dl
MODEL = "sales_dl"
print(fabric.list_datasets()[["Dataset Name"]].query("`Dataset Name` == @MODEL"))

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 21, Finished, Available, Finished, False)

  Dataset Name
0     sales_dl


## D1 · Power BI también cuenta ficheros
Límites de Direct Lake para la SKU de esta capacidad, frente a los ficheros de cada tabla.

In [ ]:
sku = dl.get_sku_size()
display(dl.get_directlake_guardrails_for_sku(sku))
for t in ["sales_clustered", "sales_partitioned"]:
    print(t, "-> ficheros:", count_files(t)[0])

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 37e144c5-1eb4-46fa-a546-2d763b329e78)

sales_clustered -> ficheros: 1
sales_partitioned -> ficheros: 5000


## D2 · ¿Se queda en Direct Lake o cae a DirectQuery?

In [ ]:
display(dl.check_fallback_reason(dataset=MODEL))

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 484d10f2-3088-4108-9711-40e95a74465c)

## D3 · Solo carga las columnas que usas (transcoding)
Refrescamos para vaciar la memoria, lanzamos una consulta con `amount` y `order_date`, y miramos qué columnas están residentes.

In [ ]:
fabric.refresh_dataset(MODEL)          # el refresco reencuadra y saca las columnas de memoria
time.sleep(10)

q = '''EVALUATE SUMMARIZECOLUMNS(sales_clustered[order_date], "importe", SUM(sales_clustered[amount]))'''
fabric.evaluate_dax(MODEL, q).head()

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 24, Finished, Available, Finished, False)

,sales_clustered[order_date],[importe]
0,2025-10-01,2054482.31
1,2025-06-27,1993641.31
2,2025-03-23,2072039.2
3,2026-09-25,2044491.23
4,2024-12-17,2047222.06


In [ ]:
cols = fabric.evaluate_dax(MODEL, "EVALUATE INFO.STORAGETABLECOLUMNS()")
keep = [c for c in cols.columns if any(k in c for k in ("DIMENSION_NAME", "ATTRIBUTE_NAME", "DICTIONARY_ISRESIDENT"))]
res = cols[keep]
res = res[~res[keep[1]].astype(str).str.startswith("RowNumber")]
display(res.sort_values(keep[0]))

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ee38667a-2535-45ae-b6d9-d53d22e88eb7)

In [ ]:
spark.sql("RESTORE TABLE sales_clustered TO VERSION AS OF 2")
print(spark.table("sales_clustered").count())   # tiene que dar 6000000
fabric.refresh_dataset(MODEL)
spark.sql("DROP TABLE IF EXISTS sales_small")

StatementMeta(, af6bf911-beb8-411b-8522-9232ad9031dd, 3, Finished, Available, Finished, False)

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales_clustered` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 1 pos 0

## D4 · Refrescar es mover un puntero (framing)
Añadimos filas en Spark: el log crea una versión nueva. El modelo la ve cuando se reencuadra.

In [ ]:
contar = '''EVALUATE ROW("filas", COUNTROWS(sales_clustered))'''
print("modelo antes  :", fabric.evaluate_dax(MODEL, contar).iloc[0, 0])

gen(100_000).write.format("delta").mode("append").saveAsTable("sales_clustered")
display(spark.sql("DESCRIBE HISTORY sales_clustered").select("version", "operation").limit(2))

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 26, Finished, Available, Finished, False)

modelo antes  : 6000000


SynapseWidget(Synapse.DataFrame, c22b04bf-fd18-4b03-bca7-3e34357cd67b)

In [ ]:
t = time.time()
fabric.refresh_dataset(MODEL)
time.sleep(5)
print("modelo después:", fabric.evaluate_dax(MODEL, contar).iloc[0, 0], f"(refresco lanzado en {round(time.time() - t, 1)} s)")

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 27, Finished, Available, Finished, False)

modelo después: 6000000 (refresco lanzado en 5.6 s)


## Extra · V-Order en la tabla que lee Power BI

In [ ]:
print("V-Order por defecto en la sesión:", spark.conf.get("spark.sql.parquet.vorder.default", "no definido"))
spark.sql("ALTER TABLE sales_clustered SET TBLPROPERTIES ('delta.parquet.vorder.enabled' = 'true')")
display(spark.sql("SHOW TBLPROPERTIES sales_clustered").where("key LIKE '%vorder%'"))

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 28, Finished, Available, Finished, False)

V-Order por defecto en la sesión: false


SynapseWidget(Synapse.DataFrame, 3d31dc0b-ce64-435e-bf8d-6ae5c1162e40)

## Limpieza

In [ ]:
for t in ["sales_demo", "sales_partitioned", "sales_clustered", "sales_zorder"]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

StatementMeta(, 60ad2a64-e688-4030-959e-679592e85c23, 29, Finished, Available, Finished, False)